# Phase 3 — Exploratory Data Analysis (EDA)

## Objective

The objective of this phase is to perform a structured EDA on `merged_data` and understand fraud distribution, transaction behavior, missingness, identity coverage, cardinality, outliers, time behavior, correlations, and potential risks.

Only analysis is performed. No imputation, encoding, scaling, feature selection, model training, or deployment is performed.

## 1. EDA Objective and Scope

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 100)
sns.set_theme(style="whitegrid")
print("EDA scope: inspect and explain merged_data only.")
print("Transformations such as filling, encoding, scaling, and feature selection: NOT PERFORMED")

**Interpretation:** The notebook is intentionally descriptive. Any recommendations at the end are plans for the next phase, not implemented changes.

## 2. Load merged_data

In [ ]:
from pathlib import Path

project = Path.cwd().parent
merged_path = project / "data" / "processed" / "merged_data.csv.gz"
merged_data = pd.read_csv(merged_path, compression="gzip", low_memory=False)
print("Loaded:", merged_path)
print("Shape:", merged_data.shape)

## 3. Dataset Overview

In [ ]:
print("Rows:", len(merged_data))
print("Columns:", len(merged_data.columns))
print("Memory usage:", round(merged_data.memory_usage(deep=True).sum() / 1024**2, 2), "MB")
print("Head:")
display(merged_data.head())
print("Data types:")
display(merged_data.dtypes.value_counts())
numeric_columns = merged_data.select_dtypes(include=np.number).columns.tolist()
numeric_features = [c for c in numeric_columns if c != "isFraud"]
categorical_columns = merged_data.select_dtypes(include=["object", "category"]).columns.tolist()
print("Numerical columns:", len(numeric_columns))
print("Numerical features excluding target:", len(numeric_features))
print("Categorical columns:", len(categorical_columns))

**Interpretation:** The result should agree with Phase 2: 590,540 rows and 434 columns. The merged dataset remains wide, so later modeling will need memory-aware processing.

## 4. Target Distribution

In [ ]:
target_counts = merged_data["isFraud"].value_counts().sort_index()
target_percent = merged_data["isFraud"].value_counts(normalize=True).sort_index() * 100
fraud_count = int(target_counts.get(1, 0))
legitimate_count = int(target_counts.get(0, 0))
print("Counts:")
display(target_counts)
print("Percentages:")
display(target_percent)
print("Total Transactions:", len(merged_data))
print("Legitimate:", legitimate_count)
print("Fraud:", fraud_count)
print("Imbalance Ratio (legitimate/fraud):", legitimate_count / fraud_count)

In [ ]:
sns.countplot(data=merged_data, x="isFraud")
plt.title("Fraud vs Legitimate Transactions")
plt.xlabel("isFraud")
plt.ylabel("Number of transactions")
plt.show()

**Interpretation:** Fraud is the minority class. Accuracy can be misleading in this setting; later model evaluation should emphasize precision, recall, F1, and PR-AUC.

## 5. Transaction Amount Analysis

In [ ]:
amount_summary = merged_data["TransactionAmt"].describe()
display(amount_summary)
print("Mean:", merged_data["TransactionAmt"].mean())
print("Median:", merged_data["TransactionAmt"].median())
print("Standard deviation:", merged_data["TransactionAmt"].std())
print("Quantiles:")
display(merged_data["TransactionAmt"].quantile([0.01, 0.25, 0.50, 0.75, 0.90, 0.99, 0.999]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(merged_data["TransactionAmt"], bins=100, ax=axes[0])
axes[0].set_title("Transaction Amount Distribution")
axes[0].set_xlabel("TransactionAmt")
axes[0].set_ylabel("Count")
sns.histplot(np.log1p(merged_data["TransactionAmt"]), bins=100, ax=axes[1], color="darkorange")
axes[1].set_title("Log-Scale View of Transaction Amount")
axes[1].set_xlabel("log1p(TransactionAmt)")
axes[1].set_ylabel("Count")
plt.tight_layout()
plt.show()

**Interpretation:** The second plot is only a visualization aid; no transformed feature is created or saved.

## 6. Transaction Amount by Fraud Class

In [ ]:
amount_by_fraud = merged_data.groupby("isFraud")["TransactionAmt"].agg(["count", "mean", "median", "std", "min", "max"])
display(amount_by_fraud)
display(merged_data.groupby("isFraud")["TransactionAmt"].quantile([0.25, 0.75, 0.90, 0.99]).unstack())

In [ ]:
plot_data = merged_data[["isFraud", "TransactionAmt"]].copy()
plot_data["AmountForPlot"] = plot_data["TransactionAmt"].clip(upper=plot_data["TransactionAmt"].quantile(0.99))
sns.boxplot(data=plot_data, x="isFraud", y="AmountForPlot")
plt.title("Transaction Amount by Fraud Class (capped at 99th percentile for display)")
plt.xlabel("isFraud")
plt.ylabel("TransactionAmt")
plt.show()

**Interpretation:** Differences in group summaries or boxplots are exploratory associations. They do not by themselves prove that transaction amount is predictive or causal.

## 7. Numerical Feature Inventory and Summary

In [ ]:
numeric_summary = merged_data[numeric_features].describe().T
numeric_summary["Missing %"] = merged_data[numeric_features].isna().mean() * 100
numeric_summary["Unique Values"] = merged_data[numeric_features].nunique(dropna=False)
print("Number of numerical features:", len(numeric_features))
print("Most missing numerical features:")
display(numeric_summary.sort_values("Missing %", ascending=False).head(15))
print("Numerical features with the largest ranges:")
numeric_summary["Range"] = numeric_summary["max"] - numeric_summary["min"]
display(numeric_summary.sort_values("Range", ascending=False).head(15))

**Interpretation:** The summary identifies features for focused inspection without producing hundreds of unreadable plots.

## 8. Selected Numerical Features vs Fraud

In [ ]:
selected_numeric = [c for c in ["TransactionAmt", "TransactionDT", "card1", "card2", "card3", "card5", "addr1", "addr2"] if c in merged_data.columns]
selected_numeric_stats = merged_data.groupby("isFraud")[selected_numeric].agg(["mean", "median", "std"])
display(selected_numeric_stats)

In [ ]:
for feature in [c for c in ["TransactionAmt", "TransactionDT"] if c in merged_data.columns]:
    sample = merged_data[["isFraud", feature]].dropna()
    sample = sample.sample(n=min(100000, len(sample)), random_state=42)
    sns.boxplot(data=sample, x="isFraud", y=feature)
    plt.title(f"{feature} by Fraud Class (sampled for display)")
    plt.xlabel("isFraud")
    plt.ylabel(feature)
    plt.show()

## 9. Categorical Feature Inventory and Cardinality

In [ ]:
print("Number of categorical features:", len(categorical_columns))
cardinality = pd.DataFrame({
    "Feature": categorical_columns,
    "Unique Values": [merged_data[c].nunique(dropna=False) for c in categorical_columns],
    "Missing %": [merged_data[c].isna().mean() * 100 for c in categorical_columns]
})
cardinality["Cardinality"] = pd.cut(cardinality["Unique Values"], bins=[-1, 10, 100, np.inf], labels=["Low", "Medium", "High"])
display(cardinality.sort_values("Unique Values", ascending=False))

**Interpretation:** Cardinality is descriptive here. High-cardinality fields may need a careful encoding strategy later; no encoding is performed now.

## 10. Selected Categorical Features vs Fraud

In [ ]:
for feature in [c for c in ["ProductCD", "card4", "card6", "P_emaildomain", "R_emaildomain"] if c in merged_data.columns]:
    rates = merged_data.groupby(feature, dropna=False)["isFraud"].agg(["count", "mean"])
    rates["Fraud Rate %"] = rates["mean"] * 100
    print("\nFeature:", feature)
    display(rates.sort_values("count", ascending=False).head(15))

In [ ]:
if "ProductCD" in merged_data.columns:
    product_rate = merged_data.groupby("ProductCD", dropna=False)["isFraud"].mean().mul(100).sort_values(ascending=False)
    product_rate.plot(kind="bar", figsize=(8, 4), title="Fraud Rate by ProductCD")
    plt.xlabel("ProductCD")
    plt.ylabel("Fraud Rate (%)")
    plt.show()

**Interpretation:** Category fraud-rate differences are exploratory and may reflect cohort or data-collection effects. They are not causal conclusions.

## 11. Missing Value Analysis

In [ ]:
missing_count = merged_data.isnull().sum()
missing_percent = merged_data.isnull().mean() * 100
missing_table = pd.DataFrame({"Feature": merged_data.columns, "Missing Count": missing_count.values, "Missing %": missing_percent.values})
print("Overall missing cells:", int(missing_count.sum()))
print("Features with any missingness:", int((missing_count > 0).sum()))
display(missing_table.sort_values("Missing %", ascending=False).head(25))

In [ ]:
top_missing = missing_table.sort_values("Missing %", ascending=False).head(20)
sns.barplot(data=top_missing, y="Feature", x="Missing %", color="steelblue")
plt.title("Top 20 Features by Missing Percentage")
plt.xlabel("Missing Percentage")
plt.ylabel("Feature")
plt.show()

**Interpretation:** Missingness is a central characteristic of the integrated dataset. Missing values are not filled, and features are not removed in EDA.

## 12. Missingness vs Fraud and Identity Availability

In [ ]:
missingness_results = []
for feature in [c for c in ["TransactionAmt", "ProductCD", "card2", "P_emaildomain", "DeviceType", "id_01", "id_02", "id_33"] if c in merged_data.columns]:
    rates = merged_data.groupby(merged_data[feature].isna())["isFraud"].agg(["count", "mean"])
    rates["Fraud Rate %"] = rates["mean"] * 100
    rates["Feature"] = feature
    rates["Status"] = rates.index.map({True: "Missing", False: "Present"})
    missingness_results.append(rates.reset_index(drop=True))
missingness_vs_fraud = pd.concat(missingness_results, ignore_index=True)
display(missingness_vs_fraud[["Feature", "Status", "count", "Fraud Rate %"]])

In [ ]:
identity_feature_columns = [c for c in ["DeviceType", "id_01", "id_02", "id_33"] if c in merged_data.columns]
identity_available = merged_data[identity_feature_columns].notna().any(axis=1)
identity_fraud = merged_data.assign(identity_available=identity_available).groupby("identity_available")["isFraud"].agg(["count", "sum", "mean"])
identity_fraud["Fraud Rate %"] = identity_fraud["mean"] * 100
display(identity_fraud)
print("Identity availability rule: at least one of the selected identity/device fields is non-missing.")

**Interpretation:** Missingness may carry predictive information. The identity availability rule is explicit and based on any populated identity/device field, rather than an arbitrary single column. This is an association, not causation.

## 13. Identity Information Analysis

In [ ]:
print("Transactions with Identity:", int(identity_available.sum()))
print("Transactions without Identity:", int((~identity_available).sum()))
print("Identity Coverage %:", identity_available.mean() * 100)
print("Fraud Rate with Identity %:", merged_data.loc[identity_available, "isFraud"].mean() * 100)
print("Fraud Rate without Identity %:", merged_data.loc[~identity_available, "isFraud"].mean() * 100)
identity_fraud["Fraud Rate %"].plot(kind="bar", title="Identity Availability vs Fraud Rate")
plt.xlabel("Identity available")
plt.ylabel("Fraud Rate (%)")
plt.show()

## 14. Outlier Analysis

In [ ]:
outlier_rows = []
for feature in [c for c in ["TransactionAmt", "TransactionDT", "card1", "card2", "addr1", "addr2"] if c in merged_data.columns]:
    values = merged_data[feature].dropna()
    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = int(((values < lower) | (values > upper)).sum())
    outlier_rows.append({"Feature": feature, "Q1": q1, "Q3": q3, "IQR": iqr, "Lower Bound": lower, "Upper Bound": upper, "Potential Outliers": count})
outlier_table = pd.DataFrame(outlier_rows)
display(outlier_table)

**Interpretation:** IQR flags are screening signals only. Extreme financial transactions may be legitimate or useful for fraud detection, so no outliers are removed.

## 15. Time-related Analysis

In [ ]:
if "TransactionDT" in merged_data.columns:
    print("TransactionDT exists.")
    print("Range:", merged_data["TransactionDT"].min(), "to", merged_data["TransactionDT"].max())
    display(merged_data["TransactionDT"].describe())
    time_bins = pd.qcut(merged_data["TransactionDT"], q=10, duplicates="drop")
    time_fraud = merged_data.groupby(time_bins, observed=False)["isFraud"].agg(["count", "mean"])
    time_fraud["Fraud Rate %"] = time_fraud["mean"] * 100
    display(time_fraud)
    time_fraud["Fraud Rate %"].plot(kind="line", marker="o", title="Fraud Rate Across Relative TransactionDT Bins")
    plt.xlabel("Relative TransactionDT bin")
    plt.ylabel("Fraud Rate (%)")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("No TransactionDT field is available; time analysis is not applicable.")

**Interpretation:** `TransactionDT` is treated as a relative time offset, not converted into artificial calendar timestamps. Quantile bins show relative patterns only.

## 16. Correlation Analysis

In [ ]:
corr_sample = merged_data[["isFraud"] + numeric_features].sample(n=min(100000, len(merged_data)), random_state=42)
correlation = corr_sample.corr()
print("Strongest absolute correlations among numerical features (sample):")
pairs = correlation.where(np.triu(np.ones(correlation.shape), k=1).astype(bool)).stack().reset_index()
pairs.columns = ["Feature 1", "Feature 2", "Correlation"]
pairs["Absolute Correlation"] = pairs["Correlation"].abs()
display(pairs.sort_values("Absolute Correlation", ascending=False).head(20))
print("Strongest exploratory numerical relationships with isFraud:")
target_correlations = correlation["isFraud"].drop("isFraud").sort_values(key=abs, ascending=False)
display(target_correlations.head(20).to_frame("Correlation with isFraud"))

**Interpretation:** Correlation can reveal redundancy or exploratory signals, but it does not prove predictive usefulness, causation, or suitability for feature selection.

## 17. Potential Data Leakage Analysis

In [ ]:
suspicious_features = [c for c in merged_data.columns if c.lower() in ["isfraud", "transactionid"] or "id" in c.lower() or "timestamp" in c.lower() or "time" in c.lower()]
print("Features requiring leakage review:")
print(suspicious_features)
print("\nReview notes:")
print("- isFraud is the target and must not be used as an input feature.")
print("- TransactionID is a direct identifier and should not automatically be treated as predictive.")
print("- TransactionDT must be available at prediction time and interpreted as a relative offset.")
print("- Identity/device fields require confirmation that they are available at prediction time.")

**Interpretation:** These are review flags, not automatic removals. Leakage decisions belong to the later preprocessing/modeling design after understanding the prediction-time scenario.

## 18. Important EDA Findings and Recommendations

In [ ]:
print("Key findings:")
print("1. The dataset remains highly imbalanced, so accuracy should not be the primary metric later.")
print("2. Missingness is extensive and identity coverage is incomplete; missingness may itself be informative.")
print("3. TransactionAmt has a wide distribution and potential extreme values; no outliers should be removed automatically.")
print("4. Categorical cardinality varies; high-cardinality fields need a deliberate later encoding strategy.")
print("5. TransactionDT is a relative time field and should not be converted to artificial dates without semantics.")
print("6. ID-like and post-transaction-like fields require leakage review before modeling.")
print("\nRecommendations for Phase 4:")
print("- Fit missing-value handling on training data only.")
print("- Compare simple categorical encodings appropriate to cardinality.")
print("- Keep a representative validation/test distribution.")
print("- Investigate, rather than automatically remove, outliers and suspicious identifiers.")
print("- Use a memory-aware preprocessing pipeline.")

## 19. Phase 3 — Final EDA Report

In [ ]:
print("PHASE 3 — FINAL EDA REPORT")
print("=" * 40)
print("1. Dataset Overview")
print("Rows:", len(merged_data))
print("Columns:", len(merged_data.columns))
print("Numerical Features:", len(numeric_features))
print("Categorical Features:", len(categorical_columns))
print("\n2. Target")
print("Fraud Count:", fraud_count)
print("Legitimate Count:", legitimate_count)
print("Fraud %:", target_percent.get(1, 0))
print("Imbalance Ratio:", legitimate_count / fraud_count)
print("\n3. Missing Values")
print("Overall Missing Cells:", int(missing_count.sum()))
print("Most Missing Features:", missing_table.sort_values("Missing %", ascending=False).head(10)["Feature"].tolist())
print("Important Pattern: identity fields are missing for transactions without identity records.")
print("\n4. Identity")
print("Identity Coverage %:", identity_available.mean() * 100)
print("Fraud Rate With Identity %:", merged_data.loc[identity_available, "isFraud"].mean() * 100)
print("Fraud Rate Without Identity %:", merged_data.loc[~identity_available, "isFraud"].mean() * 100)
print("\n5. Numerical Features")
print("TransactionAmt mean/median:", merged_data["TransactionAmt"].mean(), "/", merged_data["TransactionAmt"].median())
print("\n6. Categorical Features")
print("Highest-cardinality categorical features:")
display(cardinality.sort_values("Unique Values", ascending=False).head(10))
print("\n7. Cardinality")
print("High-cardinality categorical features:", cardinality.loc[cardinality["Cardinality"] == "High", "Feature"].tolist())
print("\n8. Outliers")
display(outlier_table[["Feature", "Potential Outliers"]])
print("No outliers removed.")
print("\n9. Time Patterns")
print("TransactionDT available:", "TransactionDT" in merged_data.columns)
print("\n10. Correlation")
display(pairs.sort_values("Absolute Correlation", ascending=False).head(5))
print("\n11. Potential Leakage")
print("Review TransactionID, isFraud, TransactionDT, and identity/device fields before modeling.")
print("\n12. Key Findings")
print("Imbalance, missingness, partial identity coverage, high-cardinality fields, wide distributions, and leakage review are the main findings.")
print("\n13. Preprocessing Recommendations")
print("Use train-only missing-value handling and encoding decisions; investigate outliers and identifiers before modeling.")
print("\nDECISION: Phase 3 EDA is complete. Do not start 04_preprocessing.ipynb until explicit approval.")

**STOP:** This notebook completes Phase 3 only. No preprocessing, encoding, scaling, feature selection, modeling, database, API, dashboard, or deployment was performed.